# Split Patient-Aware do Dataset ERCP

Divide o dataset ao nível do paciente para garantir que não há data leakage entre train/val/test.

Editar as variáveis de configuração antes de correr.

### Imports

In [ ]:
import os
import shutil
import pandas as pd
from sklearn.model_selection import train_test_split

# libs de visualização (opcional)
import matplotlib.pyplot as plt
try:
    import seaborn as sns
except Exception:
    sns = None

try:
    from IPython.display import display
except Exception:
    display = lambda x: print(x)

### Configuração

In [ ]:
# configuração — ajustar aqui
BASE_PATH = os.getenv("BASE_PATH", "../dataset")
CSV_PATH = os.getenv("CSV_PATH", os.path.join(BASE_PATH, "metadata.csv"))
PROCESSED_DIR = os.getenv("PROCESSED_DIR", os.path.join(BASE_PATH, "processed"))
OUTPUT_BASE = os.getenv("OUTPUT_BASE", "../dataset/split")

TRAIN_RATIO = float(os.getenv("TRAIN_RATIO", 0.7))
VAL_RATIO = float(os.getenv("VAL_RATIO", 0.15))
TEST_RATIO = float(os.getenv("TEST_RATIO", 0.15))
RANDOM_STATE = int(os.getenv("RANDOM_STATE", 42))

# DRY_RUN=True para ver o que seria feito sem escrever no disco
DRY_RUN = os.getenv("DRY_RUN", "False").lower() in ("1", "true", "yes")

assert abs((TRAIN_RATIO + VAL_RATIO + TEST_RATIO) - 1.0) < 1e-6, "TRAIN+VAL+TEST deve somar 1.0"

print(f"BASE_PATH={BASE_PATH}\nCSV_PATH={CSV_PATH}\nOUTPUT_BASE={OUTPUT_BASE}\nTRAIN/VAL/TEST={TRAIN_RATIO}/{VAL_RATIO}/{TEST_RATIO}\nRANDOM_STATE={RANDOM_STATE}\nDRY_RUN={DRY_RUN}")

### Função auxiliar para nomes de pastas

In [ ]:
def sanitize_label(s):
    # converte o nome da label num nome de pasta válido
    return "".join(c if str(c).isalnum() or c in "._-" else "_" for c in str(s).strip())

print(sanitize_label('Malignant Stricture'))
print(sanitize_label('Normal'))

### Carregar o CSV com os metadados

In [ ]:
if not os.path.exists(CSV_PATH):
    print(f"CSV não encontrado em {CSV_PATH}. Verificar o BASE_PATH.")
    df = None
else:
    df = pd.read_csv(CSV_PATH)
    print(f'DataFrame carregado: {df.shape}')
    display(df.head())
    print('\nColunas disponíveis:', [c for c in ['patient_id', 'Label', 'image_type', 'Keep', 'processed_image_path'] if c in df.columns])

### Filtrar linhas relevantes

In [ ]:
if df is None:
    print('Sem dados.')
else:
    before = df.shape[0]
    df = df[(df['Keep'] == 'Keep') & (df['Label'] != 'Unlabelled')].copy()
    after = df.shape[0]
    print(f'Linhas antes={before}, depois={after}')
    print(df['Label'].value_counts())

### Consolidar labels de Stricture (Malignant + Benign → Stricture)

In [ ]:
if df is None:
    print('Sem dados.')
else:
    df['Label'] = df['Label'].replace({
        'Malignant Stricture': 'Stricture',
        'Benign Stricture': 'Stricture'
    })
    print('Distribuição após merge:')
    print(df['Label'].value_counts())

### Calcular label dominante por paciente

In [ ]:
if df is None:
    print('Sem dados.')
else:
    combo_counts = (
        df.groupby(['patient_id', 'Label', 'image_type']).size().reset_index(name='count')
    )
    patient_df = (
        combo_counts.sort_values(['patient_id', 'count'], ascending=[True, False])
        .drop_duplicates('patient_id')
        .reset_index(drop=True)
    )
    patient_df['combo'] = patient_df['Label'].astype(str) + '__' + patient_df['image_type'].astype(str)
    print('Label dominante por paciente:')
    display(patient_df.head())

### Dividir pacientes em train / val / test

In [ ]:
if df is None:
    print('Sem dados.')
else:
    try:
        train_patients, temp_patients = train_test_split(
            patient_df,
            test_size=(1 - TRAIN_RATIO),
            stratify=patient_df['combo'],
            random_state=RANDOM_STATE,
        )
    except ValueError:
        print("Aviso: stratify por combo falhou, a usar stratify por Label.")
        train_patients, temp_patients = train_test_split(
            patient_df,
            test_size=(1 - TRAIN_RATIO),
            stratify=patient_df['Label'],
            random_state=RANDOM_STATE,
        )

    try:
        val_patients, test_patients = train_test_split(
            temp_patients,
            test_size=TEST_RATIO / (VAL_RATIO + TEST_RATIO),
            stratify=temp_patients['combo'],
            random_state=RANDOM_STATE,
        )
    except ValueError:
        print("Aviso: stratify por combo falhou no temp split, a usar stratify por Label.")
        val_patients, test_patients = train_test_split(
            temp_patients,
            test_size=TEST_RATIO / (VAL_RATIO + TEST_RATIO),
            stratify=temp_patients['Label'],
            random_state=RANDOM_STATE,
        )

    print('Pacientes: train =', len(train_patients), '| val =', len(val_patients), '| test =', len(test_patients))

### Atribuir split a cada linha do DataFrame

In [ ]:
if df is None:
    print('Sem dados.')
else:
    splits = {
        'train': set(train_patients['patient_id']),
        'val': set(val_patients['patient_id']),
        'test': set(test_patients['patient_id'])
    }

    def patient_split(pid):
        for s, patients in splits.items():
            if pid in patients:
                return s
        return None

    df['split'] = df['patient_id'].apply(patient_split)
    print('Contagem por split:')
    print(df['split'].value_counts(dropna=False))

### Criar estrutura de pastas

In [ ]:
if df is None:
    print('Sem dados.')
else:
    labels = sorted(df['Label'].unique())
    label_to_dir = {lbl: sanitize_label(lbl) for lbl in labels}

    for split in ['train', 'val', 'test']:
        for label, dir_name in label_to_dir.items():
            out_dir = os.path.join(OUTPUT_BASE, split, dir_name)
            if DRY_RUN:
                print('DRY_RUN: criaria', out_dir)
            else:
                os.makedirs(out_dir, exist_ok=True)
    print('Pastas criadas.')

### Copiar imagens para as pastas do split

In [ ]:
copied = 0
skipped = 0
missing = 0

if df is None:
    print('Sem dados.')
else:
    for _, row in df.iterrows():
        pid = row['patient_id']
        split = row.get('split')
        if not split:
            skipped += 1
            continue

        src = os.path.join(BASE_PATH, row['processed_image_path']) if 'processed_image_path' in row else os.path.join(PROCESSED_DIR, row.get('filename', ''))
        label_dir = label_to_dir.get(row['Label'], sanitize_label(row['Label']))
        dst_name = f"{pid}_{os.path.basename(row.get('processed_image_path', ''))}"
        dst = os.path.join(OUTPUT_BASE, split, label_dir, dst_name)

        if os.path.exists(src):
            if DRY_RUN:
                copied += 1
            else:
                try:
                    shutil.copy2(src, dst)
                    copied += 1
                except Exception as e:
                    print('Erro ao copiar', src, '->', dst, ':', e)
        else:
            missing += 1

    print(f"copiadas={copied}, em falta={missing}, ignoradas={skipped}")

In [ ]:
import cv2
from pathlib import Path
from tqdm.notebook import tqdm


def resize_and_pad(image_path, size=512, write=True):
    # redimensiona mantendo aspect ratio e faz padding a preto para ficarem quadradas
    image_path = Path(image_path)
    img = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if img is None:
        print(f"Erro ao carregar: {image_path}")
        return False

    h, w = img.shape[:2]
    scale = size / max(h, w)
    new_w, new_h = max(1, int(w * scale)), max(1, int(h * scale))
    interp = cv2.INTER_AREA if scale < 1 else cv2.INTER_CUBIC
    resized = cv2.resize(img, (new_w, new_h), interpolation=interp)

    top_pad = (size - new_h) // 2
    bottom_pad = size - new_h - top_pad
    left_pad = (size - new_w) // 2
    right_pad = size - new_w - left_pad

    padded = cv2.copyMakeBorder(
        resized, top_pad, bottom_pad, left_pad, right_pad,
        cv2.BORDER_CONSTANT, value=[0, 0, 0]
    )

    if padded.shape[0] != size or padded.shape[1] != size:
        padded = cv2.resize(padded, (size, size), interpolation=cv2.INTER_AREA)

    if write:
        cv2.imwrite(str(image_path), padded)
    return True


if df is None:
    print('Sem dados.')
else:
    candidates = []
    for split in ['train', 'val', 'test']:
        split_dir = Path(OUTPUT_BASE) / split
        if not split_dir.exists():
            print(f"Pasta não encontrada: {split_dir}")
            continue
        for p in split_dir.rglob('*'):
            if p.is_file() and p.suffix.lower() in {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}:
                candidates.append(p)

    print(f"Imagens a processar: {len(candidates)}  (DRY_RUN={DRY_RUN})")

    processed = 0
    failed = 0
    for p in tqdm(candidates, desc='pad+resize'):
        try:
            ok = resize_and_pad(p, size=512, write=not DRY_RUN)
            if ok:
                processed += 1
            else:
                failed += 1
        except Exception as e:
            failed += 1
            print(f"Erro em {p}: {e}")

    print(f"Concluído. OK={processed}, Erros={failed}")

### Verificações de sanidade

In [ ]:
if df is None:
    print('Sem dados.')
else:
    overlaps = (
        len(splits['train'] & splits['val']) +
        len(splits['train'] & splits['test']) +
        len(splits['val'] & splits['test'])
    )
    if overlaps > 0:
        print(f"AVISO: {overlaps} pacientes em mais do que um split!")

    print('\nDistribuição por split / label:')
    for split in ['train', 'val', 'test']:
        subset_df = df[df['patient_id'].isin(splits[split])]
        print(f"\n{split.upper()}")
        print(subset_df['Label'].value_counts())

    print('\nDistribuição por image_type:')
    for split in ['train', 'val', 'test']:
        subset_df = df[df['patient_id'].isin(splits[split])]
        print(f"\n{split.upper()}")
        print(subset_df['image_type'].value_counts())

### Assertions

In [ ]:
if df is not None:
    assert abs((TRAIN_RATIO + VAL_RATIO + TEST_RATIO) - 1.0) < 1e-6
    assert len(splits['train'] & splits['val']) == 0
    assert len(splits['train'] & splits['test']) == 0
    assert len(splits['val'] & splits['test']) == 0
    print('OK — sem sobreposições entre splits.')
else:
    print('Sem dados.')

### Visualizar distribuição por split

In [ ]:
if df is None:
    print('Sem dados.')
else:
    try:
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        label_counts = df.groupby('split')['Label'].value_counts().unstack(fill_value=0)
        label_counts.plot(kind='bar', stacked=False, ax=axes[0])
        axes[0].set_title('Labels por split')

        itype_counts = df.groupby('split')['image_type'].value_counts().unstack(fill_value=0)
        itype_counts.plot(kind='bar', stacked=False, ax=axes[1])
        axes[1].set_title('Tipos de imagem por split')

        plt.tight_layout()
        plt.show()
    except Exception as e:
        print('Erro ao gerar plots:', e)

### Notas
- Colocar `DRY_RUN = False` para criar as pastas e copiar os ficheiros.
- O split é feito ao nível do paciente para evitar data leakage.